# Algoritmi Greedy: Activity Selection

Notebook pentru demonstrarea strategiei care selecteaza numarul maxim de activitati compatibile. Executati celulele in ordine.

**Ideea Greedy:** alegem activitatea compatibila care se termina cel mai devreme.

In [ ]:
from pathlib import Path
import sys

DIRECTOR_PROIECT = Path.cwd()
if not (DIRECTOR_PROIECT / 'main.py').exists():
    DIRECTOR_PROIECT = DIRECTOR_PROIECT.parent
sys.path.insert(0, str(DIRECTOR_PROIECT))

from main import Activitate, selecteaza_activitati, selectie_compatibila

## 1. Enuntul problemei

Fiecare activitate are un moment de inceput si unul de final. Vrem sa selectam cat mai multe activitati care nu se suprapun. Consideram intervale semi-deschise: daca o activitate se termina exact cand incepe urmatoarea, ele sunt compatibile.

## Date de intrare

Datele sunt definite direct in urmatorul cell Python, prin obiecte `Activitate`. Valorile sunt hardcodate pentru ca demonstratia sa poata fi rerulata identic, fara introducere manuala de la tastatura.


In [ ]:
activitati = [
    Activitate('A', 1, 4),
    Activitate('B', 3, 5),
    Activitate('C', 0, 6),
    Activitate('D', 5, 7),
    Activitate('E', 8, 9),
    Activitate('F', 5, 9),
]

print('Date de intrare:')
for activitate in activitati:
    print(f'{activitate.nume}: [{activitate.inceput}, {activitate.sfarsit})')


## 2. Aplicarea strategiei Greedy

Sortam dupa timpul de finalizare, alegem prima activitate, apoi acceptam o activitate numai daca incepe dupa sau exact la finalul ultimei activitati selectate.

In [ ]:
selectate = selecteaza_activitati(activitati)

print('Activitati selectate:')
for activitate in selectate:
    print(f'{activitate.nume}: [{activitate.inceput}, {activitate.sfarsit})')

print(f'\nNumar activitati selectate: {len(selectate)}')
print('Selectia este compatibila:', selectie_compatibila(selectate))

Rezultatul asteptat pentru acest set este **A, D, E**. Algoritmul obtine 3 activitati. Sortarea are costul dominant: O(n log n); parcurgerea dupa sortare este O(n).

## 3. De ce nu alegem pur si simplu activitatea care incepe prima?

Un interval care incepe primul poate ocupa aproape tot timpul si poate bloca mai multe activitati scurte, dar compatibile intre ele.

### Exemplu separat: cand alegerea dupa timpul de inceput esueaza

Datele de intrare sunt definite intr-un cell separat. Activitatea A incepe prima, dar ocupa aproape tot intervalul disponibil.


In [ ]:
exemplu_start = [
    Activitate('A', 1, 10),
    Activitate('B', 2, 3),
    Activitate('C', 3, 4),
    Activitate('D', 4, 5),
]


In [ ]:
def selecteaza_dupa_inceput(obiecte):
    ordonate = sorted(obiecte, key=lambda activitate: (activitate.inceput, activitate.sfarsit, activitate.nume))
    rezultat = []
    ultimul_sfarsit = None
    for activitate in ordonate:
        if ultimul_sfarsit is None or activitate.inceput >= ultimul_sfarsit:
            rezultat.append(activitate)
            ultimul_sfarsit = activitate.sfarsit
    return rezultat

print('Greedy dupa inceput:', [activitate.nume for activitate in selecteaza_dupa_inceput(exemplu_start)])
print('Greedy dupa sfarsit:', [activitate.nume for activitate in selecteaza_activitati(exemplu_start)])


## 4. Contraexemplu: durata minima

Activitatea S este mai scurta, dar se suprapune cu ambele activitati A si B. Alegerea ei reduce numarul total de activitati.

### Date de intrare pentru contraexemplul „durata minima”

Activitatea S este cea mai scurta, dar se suprapune cu A si B. Datele sunt definite separat de codul care aplica cele doua criterii.


In [ ]:
exemplu_durata = [
    Activitate('A', 0, 4),
    Activitate('B', 5, 9),
    Activitate('S', 3, 6),
]


In [ ]:
def selecteaza_dupa_durata(obiecte):
    ordonate = sorted(obiecte, key=lambda activitate: (activitate.sfarsit - activitate.inceput, activitate.sfarsit, activitate.nume))
    rezultat = []
    for activitate in ordonate:
        if selectie_compatibila(rezultat + [activitate]):
            rezultat.append(activitate)
    return rezultat

print('Greedy dupa durata:', [activitate.nume for activitate in selecteaza_dupa_durata(exemplu_durata)])
print('Greedy dupa sfarsit:', [activitate.nume for activitate in selecteaza_activitati(exemplu_durata)])


## 5. Alta problema: 0/1 Knapsack

In varianta 0/1, fiecare obiect este luat integral sau deloc. Vom compara euristica raportului valoare/greutate cu rezultatul exact pentru un exemplu mic.

### Date de intrare pentru 0/1 Knapsack

Capacitatea rucsacului este 50. Obiectele sunt indivizibile: fiecare este luat integral sau deloc. Datele sunt introduse in cell-ul urmator.


In [ ]:
from main import Obiect

obiecte = [
    Obiect('A', greutate=10, valoare=60),
    Obiect('B', greutate=20, valoare=100),
    Obiect('C', greutate=30, valoare=120),
]
capacitate = 50

print('Obiecte:')
for obiect in obiecte:
    print(f'{obiect.nume}: greutate={obiect.greutate}, valoare={obiect.valoare}, raport={obiect.valoare / obiect.greutate:.1f}')
print('Capacitate:', capacitate)


In [ ]:
from main import (
    greedy_dupa_raport,
    solutie_optima_brutala,
    valoare_totala,
    greutate_totala,
)

rezultat_greedy = greedy_dupa_raport(obiecte, capacitate)
rezultat_exact = solutie_optima_brutala(obiecte, capacitate)

print('Greedy:', [obiect.nume for obiect in rezultat_greedy],
      '| greutate =', greutate_totala(rezultat_greedy),
      '| valoare =', valoare_totala(rezultat_greedy))
print('Optim:', [obiect.nume for obiect in rezultat_exact],
      '| greutate =', greutate_totala(rezultat_exact),
      '| valoare =', valoare_totala(rezultat_exact))


**Concluzie:** Greedy este eficient si optim pentru problema Activity Selection neponderata, datorita proprietatii alegerii Greedy. Dar strategia trebuie justificata separat pentru fiecare problema. Contraexemplul 0/1 Knapsack arata ca raportul valoare/greutate nu garanteaza o solutie optima cand obiectele nu pot fi fractionate.